# 🏥 SCOGS — MedGemma 27B Extraction on an A100 (Colab, Ollama)

Runs the **Sickle Cell Outcome Grading System (SCOGS)** feature-extraction harness (P11)
against **MedGemma 27B** served by Ollama on an NVIDIA A100.

### What this measures
1. **Feature extraction** — MedGemma reads an SCD case report and proposes structured findings.
2. **§2 verbatim quote verification** — every finding must carry a quote present *character for
   character* in the note. Anything else is rejected, not down-weighted.
3. **Deterministic grading** — verified features go through the 53 SCOGS decision tables.
   The model never assigns a grade.
4. **Determinism & cost** — temperature-0 run-to-run consistency, tokens/s, seconds per note.

### What it does *not* measure
There is no gold standard yet (that is P9). Quote-verification is a **grounding** rate, not an
accuracy rate — a quote can be verbatim and still not support the value attached to it.
**Precision is decided by the hand-check in Step 10**, and that is the number the Arm B
decision actually turns on.

> ⚠️ **Provenance.** This notebook serves a *community GGUF conversion*, not Google's original
> weights. Step 6 prints the real quantization and digest, and **halts** if the conversion emits
> corrupt tokens. Do not describe results from this notebook as "full 16-bit" unless Step 6 says
> the quantization level is `F16`.

**Runtime → Change runtime type → A100 GPU (High-RAM).**

## Step 1 — Verify the GPU

In [ ]:
!nvidia-smi

import psutil

try:
    import torch
    assert torch.cuda.is_available(), (
        "CUDA GPU not detected. Runtime -> Change runtime type -> A100 GPU.")
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except ImportError:
    device_name, vram_gb = "(torch unavailable - see nvidia-smi above)", 0.0

sys_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f"\n[GPU]        {device_name}")
print(f"[VRAM]       {vram_gb:.1f} GB")
print(f"[System RAM] {sys_ram_gb:.1f} GB")

if vram_gb and vram_gb < 35:
    print("\n!!  Under ~35 GB VRAM a 27B model will spill to CPU and inference will crawl.")
    print("    Step 6 checks the actual GPU/CPU split before you spend an hour on a run.")

## Step 2 — Repository and dependencies

Only installs what is actually missing. Colab already ships `torch`, `pandas` and friends, and
with the HuggingFace backend gone this notebook needs almost nothing beyond them.

In [ ]:
import importlib.util, os, subprocess, sys

BRANCH = "p11-harness-measurement-fixes"      # <- the branch to run from

# Absolute path throughout. A relative exists() check clones the repo INTO itself
# when this cell is re-run from inside it, and the nested copy breaks pytest with
# "import file mismatch".
REPO = "/content/st_jude"
if not os.path.exists(REPO):
    !git clone --quiet https://github.com/Edward-Bae-00/st_jude.git {REPO}
%cd {REPO}
!rm -rf {REPO}/st_jude
!find {REPO} -name __pycache__ -type d -exec rm -rf {} + 2>/dev/null

!git fetch --quiet origin {BRANCH} && git checkout --quiet {BRANCH} && git reset --quiet --hard origin/{BRANCH}

def sh(*c):
    return subprocess.run(c, capture_output=True, text=True, cwd=REPO).stdout.strip()

print("running from:", sh("git", "rev-parse", "--abbrev-ref", "HEAD"),
      "@", sh("git", "rev-parse", "--short", "HEAD"))

REQUIRED = ("pytest", "pandas", "tabulate")
missing = [m for m in REQUIRED if importlib.util.find_spec(m) is None]
if missing:
    print(f"Installing: {', '.join(missing)}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    print(f"All deps present ({', '.join(REQUIRED)}) - skipping pip.")

# Fail here, not 40 minutes later at an argparse error.
help_txt = subprocess.run([sys.executable, "scripts/experiments/medgemma_extraction.py", "--help"],
                          capture_output=True, text=True, cwd=REPO).stdout
assert "--stratify" in help_txt, (
    f"The harness on branch '{BRANCH}' predates the measurement fixes (no --stratify). "
    "Point BRANCH at the branch that has them.")
print("harness supports the current flags.")

## Step 3 — Rule engine unit tests

The decision tables, schema, predicates and the §2 verifier. If this is red, nothing downstream
means anything.

In [ ]:
!pytest -q

## Step 4 — Model choice and the Drive cache

Two independent things go wrong here, and the first run hit both.

**Quantization has to fit the GPU.** `jwang580/medgemma_27b_text_it` publishes one tag: F16,
54 GB. Colab hands out an A100-SXM4-**40 GB**. That model has never fit in VRAM — the earlier run
that "worked" was Ollama offloading it into system RAM and running on the CPU at **1.6 tok/s**
for 55 minutes. Below, quantization is chosen from detected VRAM so the model actually runs on
the GPU.

**Ollama mmaps its GGUF files, so it cannot serve them from Google Drive.** Drive FUSE does not
give mmap what it needs. Drive is therefore used as *cold storage only*: the model is copied to
local disk before the daemon ever touches it, and copied back after a pull.

> Whether that round-trip beats simply re-downloading is an open question — Drive FUSE reads are
> slow and Colab's link to HuggingFace is fast. Both are timed below. If the restore is slower
> than a fresh pull, set `USE_DRIVE = False` and stop paying for the cache.

In [ ]:
import os, pathlib, shutil, subprocess, time

USE_DRIVE = True          # Drive = cold storage only; serving is always from local disk

# --- quantization must fit VRAM -------------------------------------------
try:
    import torch
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
except Exception:
    VRAM_GB = 0.0

# The F16 build is a different publisher: jwang580 is the only tag carrying full
# 16-bit weights, and it is what an 80GB A100 has the room for. Its conversion is
# a community one and is NOT verified - Step 6's tokenizer gate is what catches a
# bad one, so do not skip Step 6 on this tier.
HF_REPO = "hf.co/unsloth/medgemma-27b-text-it-GGUF"
if VRAM_GB >= 60:
    MODEL, MODEL_GB = "jwang580/medgemma_27b_text_it", 54.0   # F16, fits an 80GB A100
elif VRAM_GB >= 34:
    MODEL, MODEL_GB = f"{HF_REPO}:Q8_0", 28.7      # best quality that fits a 40GB A100
else:
    MODEL, MODEL_GB = f"{HF_REPO}:Q4_K_M", 16.5
    if VRAM_GB < 22:
        print(f"!!  {VRAM_GB:.0f} GB VRAM - even Q4_K_M will partly offload to CPU and crawl.")
print(f"VRAM {VRAM_GB:.1f} GB  ->  {MODEL}  (~{MODEL_GB} GB)")

# --- speed: pin the model, and batch if the VRAM left over allows it -------
# Each parallel slot costs its own KV cache (~2 GB at 4096 ctx for this model).
# Weights + slots must leave the compute graph room, hence the 4 GB reserve.
# Pin the context length. Ollama's default has moved between versions, and KV is
# allocated per slot - letting it pick the model default (gemma3 declares 128K)
# would blow the headroom below. Prompts run ~1,000 tokens + 1,024 generated, so
# 4096 is ample and makes the arithmetic here true rather than hopeful.
NUM_CTX = 4096
KV_GB_PER_SLOT = 2.0                               # ~0.48 MB/token x 4096 for this model
headroom_gb = max(0.0, VRAM_GB - MODEL_GB - 4.0)   # 4 GB reserve for the compute graph
NUM_PARALLEL = max(1, min(4, int(headroom_gb // KV_GB_PER_SLOT)))
os.environ['OLLAMA_KEEP_ALIVE'] = '-1'             # never unload; a reload costs minutes
os.environ['OLLAMA_NUM_PARALLEL'] = str(NUM_PARALLEL)
os.environ['OLLAMA_CONTEXT_LENGTH'] = str(NUM_CTX)
os.environ['OLLAMA_FLASH_ATTENTION'] = '1'
print(f"headroom after weights {headroom_gb:.1f} GB  ->  OLLAMA_NUM_PARALLEL={NUM_PARALLEL} "
      f"x {NUM_CTX} ctx (~{NUM_PARALLEL * KV_GB_PER_SLOT:.0f} GB KV)")
if NUM_PARALLEL == 1:
    print("    no room to batch - the run stays sequential.")

# --- serving path: LOCAL DISK, always --------------------------------------
SERVE_DIR = pathlib.Path('/root/.ollama/models')
SERVE_DIR.mkdir(parents=True, exist_ok=True)
os.environ['OLLAMA_MODELS'] = str(SERVE_DIR)        # must be set before the daemon starts
local_free = shutil.disk_usage('/root').free / 1e9
print(f"serving from {SERVE_DIR}  ({local_free:.0f} GB free on local disk)")
if local_free < MODEL_GB * 1.3:
    print(f"!!  Only {local_free:.0f} GB free locally for a ~{MODEL_GB} GB model.")

# --- cold storage ----------------------------------------------------------
DRIVE_DIR = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')                  # no-op if already mounted
    DRIVE_DIR = pathlib.Path('/content/drive/MyDrive/scogs_ollama_models')
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    drive_gb = sum(f.stat().st_size for f in DRIVE_DIR.rglob('*') if f.is_file()) / 1e9
    print(f"Drive cold storage: {DRIVE_DIR}  ({drive_gb:.1f} GB held)")
    if drive_gb > MODEL_GB * 1.5:
        print(f"\n!!  Drive holds {drive_gb:.1f} GB but this model is ~{MODEL_GB} GB - an older")
        print( "    build is still cached and will be copied to local disk for nothing.")
        print( "    Clear it first:   !rm -rf /content/drive/MyDrive/scogs_ollama_models/*")

## Step 5 — Install Ollama, start the daemon, pull the model

Every step is guarded, so re-running this cell is free: the binary is only fetched on a fresh VM,
a second daemon is never started, and the model pull is skipped when the cache already has it.

> ⚠️ `OLLAMA_MODELS`, `OLLAMA_KEEP_ALIVE` and `OLLAMA_NUM_PARALLEL` are read **once, when the
> daemon starts**. If a daemon is already up, this cell skips the start and your Step 4 changes
> do not take effect. Kill it first: `!pkill ollama`, then re-run Steps 4 and 5.

In [ ]:
import shutil, subprocess, time, urllib.request

HOST = "http://localhost:11434"

def daemon_up(host=HOST, timeout=2):
    try:
        urllib.request.urlopen(host, timeout=timeout); return True
    except Exception:
        return False

# 1. Ollama binary
if shutil.which("ollama") is None:
    print("1/4 Installing zstd + Ollama ...")
    !sudo apt-get update -qq && sudo apt-get install -y -qq zstd
    !curl -fsSL https://ollama.com/install.sh | sh
else:
    print("1/4 Ollama already installed - skipping.")

# 2. Restore from cold storage BEFORE the daemon opens anything
restored = False
if DRIVE_DIR and (DRIVE_DIR / "manifests").exists():
    t0 = time.time()
    print("2/4 Restoring model from Drive to local disk ...")
    subprocess.run(["cp", "-r", f"{DRIVE_DIR}/.", str(SERVE_DIR)], check=True)
    print(f"    restored in {time.time() - t0:.0f}s "
          f"(compare against a fresh pull before trusting the cache)")
    restored = True
else:
    print("2/4 Nothing in cold storage - will pull fresh.")

# 3. Daemon (inherits OLLAMA_MODELS from Step 4)
if daemon_up():
    print("3/4 Daemon already running - skipping.")
else:
    print("3/4 Starting Ollama daemon ...")
    ollama_proc = subprocess.Popen(["ollama", "serve"],
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(45):
        time.sleep(2)
        if daemon_up(): break
    else:
        raise RuntimeError("Ollama daemon never came up on :11434")
    print("    daemon up.")

# 4. Pull only if the restore did not already provide it
listed = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout
if MODEL.split(":")[0] in listed:
    print(f"4/4 {MODEL} present - skipping pull.")
else:
    t0 = time.time()
    print(f"4/4 Pulling {MODEL} (~{MODEL_GB} GB) ...")
    !ollama pull {MODEL}
    print(f"    pulled in {time.time() - t0:.0f}s")
    if DRIVE_DIR:
        t0 = time.time()
        print("    copying to Drive cold storage for next session ...")
        subprocess.run(["cp", "-r", f"{SERVE_DIR}/.", str(DRIVE_DIR)], check=True)
        print(f"    saved in {time.time() - t0:.0f}s")

## Step 6 — Preflight gate ⛔

**This cell can halt the notebook, and that is the point.** Two failures are invisible until they
have already ruined a run:

| Failure | Symptom if unchecked |
|:--|:--|
| A corrupt GGUF conversion emitting `[UNK_BYTE_…]` byte tokens | every quote fails verbatim match; reads as a hallucinating model |
| The model sitting on CPU instead of the A100 | ~1.6 tok/s, an hour-long run, meaningless cost numbers |

It also records the **real quantization and digest** — §2 provenance requires the digest, and the
previous run stored `model_digest: 1`, which is a file-type enum, not a hash.

Loosening the quote matcher to tolerate corrupt tokens is *not* the fix. That matcher is the
project's entire hallucination defence; the fix is to serve weights that are not broken.

In [ ]:
import json, re, subprocess, urllib.error, urllib.request

ARTIFACTS = re.compile(r"\[UNK_BYTE_|▁")
fatal = []

# ---------------------------------------------------------------- provenance
tags = json.loads(urllib.request.urlopen(f"{HOST}/api/tags").read())["models"]
entry = next((m for m in tags if m["name"].startswith(MODEL.split(":")[0])), None)
if entry is None:
    raise RuntimeError(f"{MODEL} is not in the local cache - re-run Step 5.")

det = entry.get("details", {})
QUANT = det.get("quantization_level", "unknown")
DIGEST = entry.get("digest", "")
SIZE_GB = entry.get("size", 0) / 1e9
print("PROVENANCE")
print(f"  tag            {entry['name']}")
print(f"  family/params  {det.get('family')} / {det.get('parameter_size')}")
print(f"  format/quant   {det.get('format')} / {QUANT}")
print(f"  size on disk   {SIZE_GB:.1f} GB")
print(f"  digest         {DIGEST}")
if QUANT.upper() not in {"F16", "BF16", "F32"}:
    print(f"\n  NOTE: this is {QUANT}, NOT full 16-bit. Report it as {QUANT} in any results table.")

# ------------------------------------------------- does it even fit? (cheap)
# Check this BEFORE asking the model to generate. Ollama accepts a pull it cannot
# load and then fails /api/generate with a bare HTTP 500, which says nothing.
print(f"\nCAPACITY   model {SIZE_GB:.1f} GB vs VRAM {VRAM_GB:.1f} GB")
if VRAM_GB and SIZE_GB > 0.9 * VRAM_GB:
    fatal.append(
        f"{SIZE_GB:.1f} GB of weights will not fit {VRAM_GB:.1f} GB of VRAM. Ollama will "
        f"fail the load with an opaque 500. Pull a smaller quantization instead - "
        f"hf.co/unsloth/medgemma-27b-text-it-GGUF has Q8_0 (28.7 GB) and Q4_K_M (16.5 GB) - "
        f"and `ollama rm {entry['name']}` to reclaim the cache space.")
else:
    print("  fits.")

# ------------------------------------------------- warm up, then GPU split
if not fatal:
    probe = "The patient was started on norepinephrine and FiO2 was escalated to 60%."
    req = urllib.request.Request(
        f"{HOST}/api/generate",
        data=json.dumps({
            "model": MODEL, "stream": False, "format": "json",
            "prompt": ('Copy the sentence below exactly, character for character, as JSON '
                       '{"quote": "<sentence>"}. Reply with JSON only.\n\n' + probe),
            "options": {"temperature": 0, "seed": 0, "num_predict": 128},
        }).encode(),
        headers={"Content-Type": "application/json"})
    try:
        resp = json.loads(urllib.request.urlopen(req, timeout=900).read())
    except urllib.error.HTTPError as e:
        # The body is where Ollama puts the real reason. Never swallow it.
        body = e.read().decode("utf-8", "replace")[:600]
        raise RuntimeError(f"Ollama returned HTTP {e.code} on /api/generate.\n"
                           f"Ollama said: {body}\n"
                           f"Run `!ollama logs` or check `!ollama ps` for more.") from None
    raw = resp.get("response", "")

    print("\nPLACEMENT (ollama ps)")
    subprocess.run(["ollama", "ps"])
    dur = resp.get("eval_duration", 0) or 1
    tok_s = resp.get("eval_count", 0) / (dur / 1e9)
    print(f"  probe throughput  {tok_s:.1f} completion tok/s")
    if tok_s < 5:
        fatal.append(f"Throughput {tok_s:.1f} tok/s means the model is largely on CPU. "
                     f"The PROCESSOR column above must read 100% GPU.")

    print("\nTOKENIZER INTEGRITY")
    print(f"  raw reply  {raw[:160]}")
    if ARTIFACTS.search(raw):
        fatal.append("The served GGUF emits SentencePiece byte-token wreckage "
                     "([UNK_BYTE_...] / U+2581) inside generated text. Every quote it "
                     "produces will fail verbatim verification. Switch to a clean "
                     "conversion - do NOT loosen the matcher.")
    else:
        print("  clean - no [UNK_BYTE_...] or U+2581 artifacts.")

# ------------------------------------------------------------------- verdict
print("\n" + "=" * 72)
if fatal:
    for f in fatal:
        print("FATAL: " + f)
    print("=" * 72)
    raise RuntimeError("Preflight failed - fix the above before spending an hour on a run.")
print("Preflight passed. Safe to run the harness.")
print("=" * 72)

## Step 7 — Run the extraction harness

**The unit of evaluation is the (note, outcome) pair, not the note.** `absent` is a first-class
answer, not a failure — for most notes, most of the 53 outcomes genuinely are not there, and
saying so correctly is the job. So the eval set needs outcomes that are present *and* outcomes
that are not.

Random sampling gives you 40 pain crises and zero leg ulcers, so notes are **stratified by
outcome** (plan §7): a lexical seed picks notes likely to contain each target outcome. A note
seeded for ACS is still scored on all four outcomes, so it supplies one likely-positive and three
likely-negatives.

Seeds pick notes only — they never touch extraction, features or grading, so they cannot bias a
grade. They *do* bias which cases get seen, toward the lexically obvious ones. That is what
`--holdout-frac` is for: a quarter of the notes are drawn at random from the same pool, so the
size of that bias is **measured** rather than merely disclosed.

### Concurrency and what it costs you

Each request generates only ~30 completion tokens against a ~1,000-token prompt, so the run is
latency-bound: you pay per-request overhead 160 times over. Issuing several at once recovers most
of that, and Step 4 sized `OLLAMA_NUM_PARALLEL` to the VRAM left after the weights.

**But batching and the determinism metric do not mix.** A batch's composition depends on timing,
so it differs between repeat 1 and repeat 2. Batched float reductions are not bit-identical, so a
token can flip at temperature 0 for reasons that have nothing to do with the model — and
`run_to_run_consistency_pct`, which the P11 gate reads at ≥98%, cannot tell the two apart.

So the cell below picks one:

| `MEASURE_CONSISTENCY` | concurrency | What you get |
|:--|:--|:--|
| `True` (default) | 1 | A consistency number the gate can actually use. Slower. |
| `False` | `NUM_PARALLEL` | Throughput, cost and grounding numbers, several times faster. Consistency is confounded and the result file records that. |

Grounding, grades and cost are unaffected either way — only the consistency number is at stake.

In [ ]:
# True  -> concurrency 1: the consistency number stays meaningful (P11 gate).
# False -> batch it: several times faster, consistency confounded (recorded in provenance).
MEASURE_CONSISTENCY = True

CONCURRENCY = 1 if MEASURE_CONSISTENCY else NUM_PARALLEL
print(f"running with --concurrency {CONCURRENCY}"
      + ("" if CONCURRENCY == 1 else "  (consistency will be flagged as confounded)"))

!python scripts/experiments/medgemma_extraction.py \
    --tier full \
    --model {MODEL} \
    --backend ollama \
    --stratify \
    --holdout-frac 0.25 \
    --notes 20 \
    --repeat 2 \
    --concurrency {CONCURRENCY} \
    --out results/a100_27b_ollama.json

## Step 8 — Results

Grounding is reported against **both denominators, always together**. A finding with no quote is a
*prompt-compliance* failure, not a grounding result; quoting one number without the other is how a
run reads as either 7% or 96% depending on which you pick.

In [ ]:
import json, os
import pandas as pd

RESULT_PATH = "results/a100_27b_ollama.json"
assert os.path.exists(RESULT_PATH), "No result file - run Step 7 first."

with open(RESULT_PATH, encoding="utf-8") as f:
    data = json.load(f)

prov, prof = data["provenance"], data["profiling"]
met, runs = data["automated_metrics"], data["runs"]

print("=" * 74)
print(f"MedGemma extraction report   tier={prov['tier']}  cohort={prov.get('cohort', 'loose')}")
print(f"weights={prov['weights']}   served_as={prov['served_as']}   backend={prov['backend']}")
print(f"digest={prov.get('model_digest')}   {prov['timestamp']}")
print("=" * 74)

prop, quoted = met["proposed"], met["quoted"]
print(f"\nGROUNDING   ({prop} proposed findings)")
print(f"  null placeholders   {met['null_placeholder']:4d}   {met['null_placeholder_pct']:5.1f}% of proposed"
      + ("   <- omission rule ignored" if met['null_placeholder_pct'] > 5 else ""))
print(f"  quote verified      {met['quote_verified']:4d}   "
      f"{met['quote_verified_pct_of_quoted']:5.1f}% of quoted | {met['quote_verified_pct']:.1f}% of all")
print(f"  quote not in note   {met['quote_unfound']:4d}   "
      f"{met['hallucinated_pct_of_quoted']:5.1f}% of quoted | {met['hallucinated_quote_pct']:.1f}% of all"
      "   <- the hallucination rate")
if met.get("tokenizer_artifacts"):
    print(f"\n  !! {met['tokenizer_artifacts']} quotes carry corrupt GGUF byte tokens - "
          "these numbers are not a clean measurement.")

def band(v, good, workable):
    return "GOOD" if v >= good else ("WORKABLE" if v >= workable else "CONCERNING")

# A consistency number measured under batching cannot be read as a model property.
confounded = prov.get("consistency_confounded_by_batching", False)
consistency_verdict = ("CONFOUNDED" if confounded
                       else band(met["run_to_run_consistency_pct"], 98, 90))

gate = pd.DataFrame([
    ("Quote-verified % (of quoted)", f"{met['quote_verified_pct_of_quoted']:.1f}%",
     "≥95 / 85-95 / <85", band(met["quote_verified_pct_of_quoted"], 95, 85)),
    ("Run-to-run consistency", f"{met['run_to_run_consistency_pct']:.1f}%",
     "≥98 / 90-98 / <90", consistency_verdict),
    ("Invalid-value rate", f"{met['invalid_value_pct']:.1f}%",
     "≤2 / 2-10 / >10", band(-met["invalid_value_pct"], -2, -10)),
    ("Unparseable replies", f"{met['unparseable_replies']}",
     "0", "GOOD" if met["unparseable_replies"] == 0 else "CONCERNING"),
    ("Null-placeholder rate", f"{met['null_placeholder_pct']:.1f}%",
     "≤5", "GOOD" if met["null_placeholder_pct"] <= 5 else "CONCERNING"),
], columns=["P11 automated metric", "Value", "Good / Workable / Concerning", "Verdict"])
display(gate)
if confounded:
    print(f"!! Consistency was measured at concurrency {prov.get('concurrency')}. Batching can flip a\n"
          f"   token at temperature 0 on its own, so that number is not a model property and the\n"
          f"   P11 gate cannot use it. Re-run with MEASURE_CONSISTENCY = True to get a clean one.")

cost = pd.DataFrame([
    ("Notes × outcomes", f"{prov['notes_count']} × {len(prov['outcomes'])}"),
    ("Concurrency", f"{prov.get('concurrency', 1)}"),
    ("Wall clock", f"{prof['total_wall_clock_sec']:.0f} s"),
    ("Per note (these outcomes)", f"{prof['sec_per_note']:.1f} s"),
    ("Throughput", f"{prof['completion_tokens_per_sec']:.1f} completion tok/s"),
    ("Prompt tokens / note", f"{prof['prompt_tokens_per_note']:,}"),
    ("Extrapolated to all 53 outcomes",
     f"{prof['sec_per_note'] * 53 / len(prov['outcomes']) / 60:.0f} min/note, "
     f"{prof['prompt_tokens_per_note'] * 53 // len(prov['outcomes']):,} prompt tok/note"),
], columns=["Cost", "Value"])
display(cost)

status = data["grade_status"]
total = sum(status.values())
display(pd.DataFrame(
    [(k, v, f"{100 * v / total:.1f}%") for k, v in sorted(status.items(), key=lambda kv: -kv[1])],
    columns=["SCOGS status", "Note-outcome pairs", "Share"]))

COLS = ["graded", "grade_set", "cannot_grade", "absent", "not_applicable"]
n_notes = prov["notes_count"]

print(f"\nPER OUTCOME (n={n_notes} each) - a pooled number hides this shape entirely.")
print("An outcome with n under ~10 gets a dash, not a percentage (plan §7).")
per_out = data.get("grade_status_by_outcome", {})
display(pd.DataFrame(
    [{"outcome": f"{num} {name}", **{c: per_out.get(num, {}).get(c, 0) for c in COLS}}
     for num, name in ((n, next(o["outcome_name"] for r in data["detailed_records"]
                                for k, o in r["outcomes"].items() if k == n))
                       for n in prov["outcomes"])]))

print("\nSEEDED vs UNSTRATIFIED HOLDOUT - this is the size of the selection bias.")
print("If the holdout looks far worse, the seeds are feeding it the easy cases.")
by_sel = data.get("grade_status_by_selection", {})
if by_sel:
    rows = []
    for k, c in by_sel.items():
        n = sum(c.values())
        rows.append({"selection": k, "pairs": n,
                     **{col: f"{c.get(col, 0)} ({100 * c.get(col, 0) / n:.0f}%)" for col in COLS}})
    display(pd.DataFrame(rows))

print("\nIf most pairs are `cannot_grade`, extraction is too sparse to grade with, however\n"
      "precise the few extractions are. But a high `absent` count is NOT automatically bad -\n"
      "Step 11 is what decides whether those absences are correct.")

feats = data["features_extracted"]
display(pd.DataFrame(sorted(feats.items(), key=lambda kv: -kv[1]),
                     columns=["Feature", "Times accepted"]))

## Step 9 — Every accepted finding, with its quote

This is the audit surface: feature, value, and the exact span that justified it.

In [ ]:
import sys
sys.path.insert(0, "scripts/experiments")
from medgemma_extraction import FEATURES, coerce, normalize   # the same matcher the harness used

rows = []
for rec in data["detailed_records"]:
    note, hay = rec["patient_note"], normalize(rec["patient_note"])
    for num, o in rec["outcomes"].items():
        try:
            findings = (json.loads(o.get("raw_reply") or "{}").get("findings") or [])
        except json.JSONDecodeError:
            continue
        for f in findings:
            name, quote = f.get("feature"), f.get("quote")
            if name not in FEATURES or not quote or normalize(quote) not in hay:
                continue
            ok, val = coerce(name, f.get("value"))
            if not ok:
                continue
            gr = o["grade_result"]
            rows.append({
                "uid": rec["patient_uid"],
                "outcome": f"{num} {o['outcome_name'][:26]}",
                "feature": name,
                "value": val,
                "quote": quote[:110],
                "status": gr["status"],
                "grade": gr.get("grade"),
            })

df = pd.DataFrame(rows)
print(f"{len(df)} accepted findings across {df['uid'].nunique()} notes"
      if len(df) else "No accepted findings.")
pd.set_option("display.max_colwidth", 115)
display(df)

## Step 10 — Hand-check worksheet (the number that decides Arm B) ⭐

Quote-verification says the span is real. It does **not** say the span supports the value —
`vasopressors = true` quoting *"aggressive fluid resuscitation with lactated ringers along with
phenylephrine bolus and drip"* is verbatim, and judging it takes a human.

`tasks/medgemma_extraction_test.md` gates the decision on **precision**:

| Precision | Read |
|:--|:--|
| ≥ 90% | Arm B viable |
| 75–90% | good as a third engine, not alone |
| < 75% | a labeller, not an extractor — revisit the prompt before the model |

Fill in `supports_value` with `y` / `n`. Budget ~15–20 s per row, ~1.5 h total. At n=100 the 95%
interval is roughly ±6pp near 90%, so read the result as a band, not a point.

In [ ]:
HANDCHECK_PATH = "results/handcheck.csv"

if len(df):
    work = df.copy()
    work.insert(len(work.columns), "supports_value", "")        # <- you fill this in: y / n
    work.insert(len(work.columns), "reviewer_note", "")
    work = work.sample(n=min(100, len(work)), random_state=0)   # capped per the P11 protocol
    work.to_csv(HANDCHECK_PATH, index=False)
    print(f"Wrote {len(work)} rows to {HANDCHECK_PATH}")
    print("\nOpen it, mark supports_value as y/n, then compute:")
    print("   precision = y / (y + n)")
    print("\nAlso read 5 full notes and list what SHOULD have been extracted for the four")
    print("outcomes, then diff against the rows above. That is the recall check - directional")
    print("at n=5, which is fine: the question is whether it misses things plainly stated.")
else:
    print("Nothing accepted to hand-check. That is itself the finding - extraction is too sparse.")

## Step 11 — Absence audit (does "none" mean none?) ⭐

Step 10 measures the **present** side: of what the model extracted, how much is right. It says
nothing about what it *missed*.

Almost every `absent` verdict comes straight from the model's `present: false` flag, and nothing
in the pipeline checks it. That flag decides whether an outcome is graded at all — so it gates
everything Step 10 measures, and it is currently the least-examined decision in the system.

`tasks/plan.md` §7 names this:

> **Absence audit.** Human-confirm a random sample of (note, outcome) pairs marked *absent* →
> false-negative rate estimate. This is the number that protects the absent / cannot-determine
> distinction (§1).

Read the note, then answer one question: **is this outcome genuinely not in this note?**
`y` = correctly absent. `n` = a false negative, the outcome is there and was missed.

Sort by `uid` so you read each note once and answer for all four outcomes together — about
10 minutes per note. A false-negative rate here is the counterweight to precision: a model can
score 100% precision by extracting almost nothing, and only this catches that.

In [ ]:
from medgemma_extraction import build_prompt

ABSENCE_PATH = "results/absence_audit.csv"

def what_would_make_it_present(num):
    # the exact feature list the model was shown, so the reviewer checks the same thing
    p = build_prompt("", num)
    return " | ".join(line.strip(" -").strip()
                      for line in p.split("Extract only these findings:")[1]
                                   .split("Rules:")[0].strip().splitlines())

abs_rows = []
for rec in data["detailed_records"]:
    for num, o in rec["outcomes"].items():
        gr = o["grade_result"]
        if gr["status"] != "absent":
            continue
        abs_rows.append({
            "uid": rec["patient_uid"],
            "selection": rec.get("selection", ""),
            "outcome": num,
            "outcome_name": o["outcome_name"],
            "model_said_present": o.get("present"),
            "truly_absent": "",                      # <- you fill this in: y / n
            "reviewer_note": "",
            "look_for": what_would_make_it_present(num),
            "title": rec.get("title", ""),
            "note_text": rec["patient_note"],
        })

adf = pd.DataFrame(abs_rows)
if len(adf):
    sample = adf.sample(n=min(50, len(adf)), random_state=0).sort_values(["uid", "outcome"])
    sample.to_csv(ABSENCE_PATH, index=False)
    print(f"{len(adf)} pairs marked absent; wrote {len(sample)} to {ABSENCE_PATH}")
    print("\n  false-negative rate = n / (y + n)")
    print("  A high rate means the model is silently skipping outcomes that are present,")
    print("  which no precision number will ever reveal.")

    src = pd.DataFrame([
        ("model said present=false", int((adf["model_said_present"] == False).sum())),
        ("rules all definitively false", int((adf["model_said_present"] != False).sum())),
    ], columns=["Where the absence came from", "Pairs"])
    display(src)
    display(sample.groupby("outcome_name").size().rename("absent pairs").to_frame())
else:
    print("Nothing marked absent - unusual, and worth checking before trusting the run.")

## Step 12 — Download artifacts

In [ ]:
from google.colab import files

for path in ("results/a100_27b_ollama.json", "results/handcheck.csv",
             "results/absence_audit.csv"):
    if os.path.exists(path):
        files.download(path)
        print(f"Downloaded {path}")
    else:
        print(f"Missing {path}")